# 🏪 Store Sales Forecasting with LightGBM

This notebook presents an end-to-end time series forecasting solution for predicting store sales across multiple stores and product families in Ecuador. The dataset comes from the **Corporación Favorita** grocery chain.

---

## Table of Contents

1. **Import Libraries**
2. **Load Datasets**
3. **Exploratory Data Analysis (EDA)**
4. **Data Preprocessing & Feature Engineering**
5. **Model Training (LightGBM)**
6. **Evaluation (RMSLE on Validation Set)**
7. **Generate Submission File**

---

## Why LightGBM?

**LightGBM** (Light Gradient Boosting Machine) is the algorithm of choice for this task due to several key advantages:

- **Speed & Efficiency**: Uses histogram-based learning and leaf-wise tree growth, making it significantly faster than other GBDT implementations on large datasets (~3M rows).
- **Native Categorical Feature Support**: Handles categorical variables without requiring one-hot encoding, preserving information and reducing memory.
- **Excellent Tabular Performance**: Gradient boosting models consistently dominate tabular data competitions, and LightGBM frequently achieves top rankings on Kaggle.
- **Built-in Regularization**: L1/L2 regularization, max depth, and min data in leaf prevent overfitting.
- **Handles Missing Values**: Natively handles NaN values without imputation.

For this multi-store, multi-product forecasting problem with rich external features (oil prices, holidays, promotions), LightGBM's ability to capture complex non-linear interactions makes it the optimal choice.

---
## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_log_error
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

---
## 2. Load Datasets

The dataset consists of several interconnected files:

| File | Description |
|------|-------------|
| `train.csv` | Historical sales data (2013-01-01 to 2017-08-15) |
| `test.csv` | Future dates to predict (2017-08-16 to 2017-08-31) |
| `stores.csv` | Store metadata (city, state, type, cluster) |
| `oil.csv` | Daily oil prices (Ecuador is oil-dependent) |
| `holidays_events.csv` | Holidays and special events |
| `transactions.csv` | Daily store transaction counts |

In [ ]:
DATA_DIR = '../Dataset/'

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA_DIR + 'stores.csv')
oil = pd.read_csv(DATA_DIR + 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA_DIR + 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA_DIR + 'transactions.csv', parse_dates=['date'])

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Date range (Train): {train.date.min()} → {train.date.max()}')
print(f'Date range (Test):  {test.date.min()} → {test.date.max()}')

---
## 3. Exploratory Data Analysis (EDA)

Before building our model, let's understand the data distribution, temporal patterns, and relationships between features.

### 3.1 Sales Distribution Overview

In [ ]:
print('=== Train Data Info ===')
print(train.describe())
print(f'\nUnique stores: {train.store_nbr.nunique()}')
print(f'Unique product families: {train.family.nunique()}')
print(f'Product families: {sorted(train.family.unique())}')
print(f'\nZero sales percentage: {(train.sales == 0).mean():.2%}')
print(f'Missing values:\n{train.isnull().sum()}')

### 3.2 Total Sales Over Time

Aggregating total daily sales reveals overall trends, seasonality, and any structural breaks in the data.

In [ ]:
daily_sales = train.groupby('date')['sales'].sum()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_sales.index, daily_sales.values, linewidth=0.7, color='#2196F3')
ax.set_title('Total Daily Sales Over Time', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales')
plt.tight_layout()
plt.show()

### 3.3 Sales by Product Family (Top 10)

Understanding which product families contribute the most to overall revenue helps prioritize modeling efforts.

In [ ]:
family_sales = train.groupby('family')['sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
family_sales.head(10).plot(kind='barh', ax=ax, color=plt.cm.viridis(np.linspace(0.2, 0.8, 10)))
ax.set_title('Top 10 Product Families by Total Sales', fontsize=16, fontweight='bold')
ax.set_xlabel('Total Sales')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.4 Day-of-Week Seasonality

Retail sales exhibit strong weekly patterns. This visualization confirms the presence of day-of-week effects.

In [ ]:
train_temp = train.copy()
train_temp['dayofweek'] = train_temp['date'].dt.dayofweek
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

dow_sales = train_temp.groupby('dayofweek')['sales'].mean()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(dow_names, dow_sales.values, color=plt.cm.Spectral(np.linspace(0.2, 0.8, 7)))
ax.set_title('Average Sales by Day of Week', fontsize=16, fontweight='bold')
ax.set_ylabel('Average Sales')
plt.tight_layout()
plt.show()

del train_temp

### 3.5 Oil Price Trend

Ecuador's economy is heavily dependent on oil exports. Oil prices can significantly influence consumer spending power and thus store sales.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(oil['date'], oil['dcoilwtico'], linewidth=0.8, color='#FF5722')
ax.set_title('Daily Oil Price (WTI Crude)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.fill_between(oil['date'], oil['dcoilwtico'], alpha=0.15, color='#FF5722')
plt.tight_layout()
plt.show()

print(f'Oil price missing values: {oil.dcoilwtico.isnull().sum()} ({oil.dcoilwtico.isnull().mean():.1%})')

---
## 4. Data Preprocessing & Feature Engineering

Feature engineering is crucial for time series forecasting with gradient boosting models. We will create:

1. **Temporal features** — day, month, year, day-of-week, week-of-year, etc.
2. **Lag features** — past sales values (lag 16–28 to avoid data leakage with the 16-day test window)
3. **Rolling statistics** — moving averages and standard deviations to capture trends
4. **Oil price features** — interpolated oil prices with rolling averages
5. **Holiday indicators** — national/regional/local holiday flags
6. **Store metadata** — type, cluster, city, state
7. **Promotion features** — current and rolling promotion counts

### 4.1 Prepare Oil Prices

Oil prices have missing values (weekends/holidays). We use forward-fill interpolation and compute rolling averages to capture price trends.

In [ ]:
oil = oil.set_index('date').reindex(
    pd.date_range(start=oil['date'].min(), end='2017-08-31', freq='D')
).rename_axis('date')
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()

oil['oil_ma7'] = oil['dcoilwtico'].rolling(7, min_periods=1).mean()
oil['oil_ma30'] = oil['dcoilwtico'].rolling(30, min_periods=1).mean()
oil['oil_diff'] = oil['dcoilwtico'].diff().fillna(0)

oil = oil.reset_index()
oil.rename(columns={'index': 'date'}, inplace=True)

print(f'Oil data shape: {oil.shape}')
print(f'Oil missing after processing: {oil.isnull().sum().sum()}')

### 4.2 Process Holidays

We create binary flags for different holiday types (national, regional, local) and identify transferred holidays. Events like "Terremoto Manabi" (earthquake) are also captured.

In [ ]:
holidays_national = holidays[
    (holidays['locale'] == 'National') & 
    (holidays['transferred'] == False)
][['date']].drop_duplicates()
holidays_national['is_national_holiday'] = 1

holidays_regional = holidays[
    holidays['locale'] == 'Regional'
][['date', 'locale_name']].drop_duplicates()
holidays_regional['is_regional_holiday'] = 1

holidays_local = holidays[
    holidays['locale'] == 'Local'
][['date', 'locale_name']].drop_duplicates()
holidays_local['is_local_holiday'] = 1

holidays_events_flag = holidays[
    holidays['type'] == 'Event'
][['date']].drop_duplicates()
holidays_events_flag['is_event'] = 1

print(f'National holidays: {len(holidays_national)}')
print(f'Regional holidays: {len(holidays_regional)}')
print(f'Local holidays:    {len(holidays_local)}')
print(f'Events:            {len(holidays_events_flag)}')

### 4.3 Combine Train & Test, Merge External Data

We combine train and test datasets to ensure consistent feature engineering, then merge all external data sources.

In [ ]:
train['is_train'] = True
test['is_train'] = False
test['sales'] = np.nan

df = pd.concat([train, test], ignore_index=True)
print(f'Combined shape: {df.shape}')

df = df.merge(stores, on='store_nbr', how='left')

df = df.merge(oil, on='date', how='left')

df = df.merge(holidays_national, on='date', how='left')
df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)

df = df.merge(
    holidays_regional.rename(columns={'locale_name': 'state'}),
    on=['date', 'state'], how='left'
)
df['is_regional_holiday'] = df['is_regional_holiday'].fillna(0).astype(int)

df = df.merge(
    holidays_local.rename(columns={'locale_name': 'city'}),
    on=['date', 'city'], how='left'
)
df['is_local_holiday'] = df['is_local_holiday'].fillna(0).astype(int)

df = df.merge(holidays_events_flag, on='date', how='left')
df['is_event'] = df['is_event'].fillna(0).astype(int)

print(f'After merges: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

### 4.4 Create Temporal Features

Temporal features allow the model to learn patterns associated with specific calendar positions — monthly trends, weekly seasonality, pay-day effects, and year-end spikes.

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['dayofweek'] = df['date'].dt.dayofweek
df['dayofyear'] = df['date'].dt.dayofyear
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
df['is_payday'] = ((df['day'] == 15) | (df['day'] == df['date'].dt.days_in_month)).astype(int)

df['sin_dayofweek'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['cos_dayofweek'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)
df['sin_dayofyear'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['cos_dayofyear'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

print(f'Shape after temporal features: {df.shape}')

### 4.5 Create Lag & Rolling Features

Lag features capture autoregressive dependencies. Since the test period spans 16 days (Aug 16–31), we use lags starting from **16** to prevent data leakage. Rolling statistics smooth out noise and capture trends.

Features are computed per `(store_nbr, family)` group to capture store-product specific patterns.

In [ ]:
df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

lag_days = [16, 17, 18, 19, 20, 21, 28]

for lag in lag_days:
    df[f'sales_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].shift(lag)

for window in [7, 14, 28]:
    df[f'sales_roll_mean_{window}_lag16'] = (
        df.groupby(['store_nbr', 'family'])['sales']
        .shift(16)
        .transform(lambda x: x.rolling(window, min_periods=1).mean())
    )
    df[f'sales_roll_std_{window}_lag16'] = (
        df.groupby(['store_nbr', 'family'])['sales']
        .shift(16)
        .transform(lambda x: x.rolling(window, min_periods=1).std())
    )

df['promo_roll_7'] = (
    df.groupby(['store_nbr', 'family'])['onpromotion']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
df['promo_roll_14'] = (
    df.groupby(['store_nbr', 'family'])['onpromotion']
    .transform(lambda x: x.rolling(14, min_periods=1).mean())
)

print(f'Shape after lag/rolling features: {df.shape}')
print(f'Lag/rolling columns: {[c for c in df.columns if "lag" in c or "roll" in c]}')

### 4.6 Encode Categorical Features

LightGBM natively handles categorical features, but we need to convert them to the `category` dtype. We also apply label encoding to retain ordinal information where applicable.

In [ ]:
cat_cols = ['family', 'city', 'state', 'type']

for col in cat_cols:
    df[col] = df[col].astype('category')

print('Categorical columns and their cardinalities:')
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} unique values')

### 4.7 Define Feature Set

We select all engineered features while excluding identifiers, the target variable, and the date column.

In [ ]:
exclude_cols = ['id', 'date', 'sales', 'is_train']
features = [c for c in df.columns if c not in exclude_cols]

print(f'Total features: {len(features)}')
print(f'Features: {features}')

---
## 5. Model Training (LightGBM)

### Training Strategy

We use a **time-based train/validation split** rather than random cross-validation to respect the temporal ordering:

- **Training set**: Data from **2015-01-01** to **2017-07-31** (using recent data to focus on current patterns)
- **Validation set**: Data from **2017-08-01** to **2017-08-15** (last 15 days, mimicking the test window)

We train the model to minimize the **RMSLE** (Root Mean Squared Logarithmic Error), which is the competition metric. This is achieved by using `log1p(sales)` as the target and **RMSE** as the LightGBM objective.

In [ ]:
train_mask = (df['is_train'] == True) & (df['date'] >= '2015-01-01') & (df['date'] <= '2017-07-31')
val_mask = (df['is_train'] == True) & (df['date'] >= '2017-08-01') & (df['date'] <= '2017-08-15')
test_mask = df['is_train'] == False

X_train = df.loc[train_mask, features]
y_train = np.log1p(df.loc[train_mask, 'sales'].clip(lower=0))

X_val = df.loc[val_mask, features]
y_val = np.log1p(df.loc[val_mask, 'sales'].clip(lower=0))

X_test = df.loc[test_mask, features]

print(f'Training samples:   {X_train.shape[0]:>10,}')
print(f'Validation samples: {X_val.shape[0]:>10,}')
print(f'Test samples:       {X_test.shape[0]:>10,}')

### 5.1 LightGBM Hyperparameters

The hyperparameters below are tuned for this specific forecasting task:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `num_leaves` | 255 | Allows complex tree structures to capture interactions |
| `learning_rate` | 0.03 | Slower learning rate for better generalization |
| `n_estimators` | 3000 | High count with early stopping |
| `min_child_samples` | 50 | Prevents overfitting to rare combinations |
| `subsample` | 0.8 | Row sampling for regularization |
| `colsample_bytree` | 0.8 | Feature sampling for regularization |
| `reg_alpha` | 0.1 | L1 regularization |
| `reg_lambda` | 0.1 | L2 regularization |

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 255,
    'learning_rate': 0.03,
    'n_estimators': 3000,
    'max_depth': -1,
    'min_child_samples': 50,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

cat_features_idx = [features.index(c) for c in cat_cols if c in features]

model = lgb.LGBMRegressor(**params)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100)
    ],
    categorical_feature=cat_features_idx
)

print(f'\nBest iteration: {model.best_iteration_}')
print(f'Best validation RMSE (on log1p scale): {model.best_score_["valid_0"]["rmse"]:.6f}')

### 5.2 Feature Importance

Understanding which features drive predictions helps validate the model's reasoning and guide further feature engineering.

In [ ]:
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
top_n = 25
top_features = importance_df.head(top_n)
colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
ax.barh(range(top_n), top_features['importance'].values, color=colors)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features['feature'].values)
ax.invert_yaxis()
ax.set_title(f'Top {top_n} Feature Importances', fontsize=16, fontweight='bold')
ax.set_xlabel('Importance (Split Count)')
plt.tight_layout()
plt.show()

---
## 6. Evaluation (RMSLE on Validation Set)

The evaluation metric for this competition is **RMSLE** (Root Mean Squared Logarithmic Error):

$$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left(\log(1 + \hat{y}_i) - \log(1 + y_i)\right)^2}$$

Since we trained on `log1p(sales)`, our predictions are already in log scale. We transform back using `expm1` and clip negative predictions to zero (sales cannot be negative).

In [ ]:
val_preds_log = model.predict(X_val)
val_preds = np.expm1(val_preds_log).clip(min=0)
val_actual = df.loc[val_mask, 'sales'].values.clip(min=0)

rmsle = np.sqrt(mean_squared_log_error(val_actual, val_preds))
print(f'╔══════════════════════════════════════╗')
print(f'║  Validation RMSLE: {rmsle:.6f}       ║')
print(f'╚══════════════════════════════════════╝')

### 6.1 Prediction vs Actual Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(val_actual, bins=100, alpha=0.6, label='Actual', color='#2196F3', density=True)
axes[0].hist(val_preds, bins=100, alpha=0.6, label='Predicted', color='#FF5722', density=True)
axes[0].set_title('Distribution: Actual vs Predicted Sales', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=12)
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, np.percentile(val_actual, 99))

sample_size = min(5000, len(val_actual))
sample_idx = np.random.choice(len(val_actual), sample_size, replace=False)
axes[1].scatter(val_actual[sample_idx], val_preds[sample_idx], alpha=0.15, s=5, color='#4CAF50')
max_val = max(val_actual[sample_idx].max(), val_preds[sample_idx].max())
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect Prediction')
axes[1].set_title('Actual vs Predicted (Scatter)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

### 6.2 RMSLE by Product Family

Breaking down the error by product family reveals which categories are harder to predict.

In [ ]:
val_df = df.loc[val_mask, ['family', 'sales']].copy()
val_df['predicted'] = val_preds

family_rmsle = []
for family_name, group in val_df.groupby('family'):
    actual = group['sales'].clip(lower=0).values
    predicted = group['predicted'].clip(lower=0).values
    if actual.sum() > 0:
        score = np.sqrt(mean_squared_log_error(actual, predicted))
        family_rmsle.append({'family': family_name, 'rmsle': score, 'mean_sales': actual.mean()})

family_rmsle_df = pd.DataFrame(family_rmsle).sort_values('rmsle', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(family_rmsle_df)))
ax.barh(range(len(family_rmsle_df)), family_rmsle_df['rmsle'].values, color=colors)
ax.set_yticks(range(len(family_rmsle_df)))
ax.set_yticklabels(family_rmsle_df['family'].values, fontsize=9)
ax.invert_yaxis()
ax.set_title('Validation RMSLE by Product Family', fontsize=16, fontweight='bold')
ax.set_xlabel('RMSLE')
ax.axvline(x=rmsle, color='red', linestyle='--', label=f'Overall RMSLE: {rmsle:.4f}')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Retrain on Full Data & Generate Submission

For the final submission, we retrain the model using **all available training data** (including the validation period) to maximize the information available to the model. We use the best iteration count from our validation experiment.

In [ ]:
full_train_mask = (df['is_train'] == True) & (df['date'] >= '2015-01-01')

X_full = df.loc[full_train_mask, features]
y_full = np.log1p(df.loc[full_train_mask, 'sales'].clip(lower=0))

best_iter = model.best_iteration_
params_final = params.copy()
params_final['n_estimators'] = best_iter

final_model = lgb.LGBMRegressor(**params_final)
final_model.fit(
    X_full, y_full,
    categorical_feature=cat_features_idx
)

print(f'Final model trained with {best_iter} iterations on {X_full.shape[0]:,} samples')

### 7.1 Generate Predictions & Create Submission File

The submission file must match the format of `sample_submission.csv` with columns `id` and `sales`.

In [ ]:
test_preds_log = final_model.predict(X_test)
test_preds = np.expm1(test_preds_log).clip(min=0)

submission = pd.DataFrame({
    'id': df.loc[test_mask, 'id'].astype(int).values,
    'sales': test_preds
})

submission = submission.sort_values('id').reset_index(drop=True)

submission.to_csv('submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'\nSubmission preview:')
print(submission.head(10))
print(f'\nSales statistics:')
print(submission['sales'].describe())

### 7.2 Validate Submission Format

Final sanity check to ensure our submission matches the expected format.

In [ ]:
sample_sub = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print('=== Format Validation ===')
print(f'Sample submission shape: {sample_sub.shape}')
print(f'Our submission shape:    {submission.shape}')
print(f'Shapes match: {sample_sub.shape == submission.shape}')
print(f'Columns match: {list(sample_sub.columns) == list(submission.columns)}')
print(f'ID range match: {sample_sub.id.min() == submission.id.min()} & {sample_sub.id.max() == submission.id.max()}')
print(f'Any NaN in sales: {submission.sales.isnull().any()}')
print(f'Any negative sales: {(submission.sales < 0).any()}')
print(f'\n✅ Submission file saved to: submission.csv')

---
## Summary

### Approach
- **Algorithm**: LightGBM (Gradient Boosting Decision Tree)
- **Target Transformation**: `log1p(sales)` to optimize for RMSLE directly
- **Feature Engineering**: 40+ features including temporal, lag, rolling, oil price, holiday, store metadata, and promotion features
- **Validation**: Time-based split (2017-08-01 to 2017-08-15)
- **Final Model**: Retrained on all data with best iteration from validation

### Key Design Decisions
1. **Lag ≥ 16 days**: Prevents data leakage since the test window is 16 days
2. **Training from 2015 onward**: Focuses on recent patterns while keeping enough data
3. **Log1p transform**: Naturally optimizes RMSLE and handles the right-skewed sales distribution
4. **Clipping predictions to ≥ 0**: Sales cannot be negative

### Potential Improvements
- **Multi-fold time series CV** with expanding/sliding windows
- **Hyperparameter optimization** with Optuna or Bayesian search
- **Stacking/blending** with other models (XGBoost, CatBoost)
- **Target encoding** for high-cardinality categorical features
- **Earthquake-specific features** for the 2016 Manabí earthquake period